# 🔬 Silicon Photonics Measurement Analysis — Demo Notebook

**프로젝트:** HY202103 Silicon Photonics 측정 데이터 분석  
**분석 대상:** GPDO (Germanium Photodetector) / MZM (Mach-Zehnder Modulator)  
**지원 웨이퍼:** D07, D08, D23, D24

---
### ✅ 노트북 사용 방법
1. **Cell 1** — 환경 설정 (반드시 먼저 실행)
2. **Cell 2** — 분석 조건 선택 (웨이퍼, 소자 종류, 파일 선택)
3. **Cell 3** — 데이터 불러오기 및 XML 파싱
4. **Cell 4** — 분석 결과 표 출력
5. **Cell 5** — Figure & Graph
6. **Cell 6** — CSV / PNG 저장
7. **Cell 7 (선택)** — 웨이퍼 전체 일괄 분석

> **Tip:** `Kernel → Restart & Run All` 로 전체 실행 가능

---
## Cell 1 — 환경 설정

In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Cell 1 : 환경 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, warnings
warnings.filterwarnings('ignore')

# 프로젝트 루트를 sys.path 에 추가
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 프로젝트 모듈 import
from config import DATA_DIR, RES_DIR, PROJECT_NAME
from src.gpdo.parser    import GPDOParser
from src.gpdo.fitting   import FittingEngine
from src.mzm.parser     import MZMParser
from src.mzm.fitting    import fit_mzi, parse_array, process_spectrum, process_iv

print('━'*55)
print(f'  프로젝트  : {PROJECT_NAME}')
print(f'  데이터 경로: {DATA_DIR}')
print(f'  저장 경로  : {RES_DIR}')
print('━'*55)
print('✅ 환경 설정 완료')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  프로젝트  : HY202103
  데이터 경로: C:\Users\Hyunsoo\PycharmProjects\Project\data\HY202103
  저장 경로  : C:\Users\Hyunsoo\PycharmProjects\Project\res
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ 환경 설정 완료


---
## Cell 2 — 분석 조건 선택 (위젯)

> 아래 셀을 실행하면 드롭다운이 순서대로 나타납니다.  
> **소자 종류 → 웨이퍼 → Timestamp → Die (col, row)** 순으로 클릭하세요.

In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Cell 2 : 위젯 기반 분석 조건 선택
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── 소자별 지원 웨이퍼 매핑 ───────────────────────────
DEVICE_WAFER_MAP = {
    'GPDO': ['D08', 'D23', 'D24'],
    'MZM' : ['D07', 'D08', 'D23', 'D24'],
}

# ── 선택 상태 저장용 dict ──────────────────────────────
selection = dict(device=None, wafer=None, timestamp=None,
                 die=None, xml_path=None)

# ── 레이아웃 공통 설정 ─────────────────────────────────
dd_layout = widgets.Layout(width='420px')
style     = {'description_width': '140px'}

# ── 위젯 생성 ─────────────────────────────────────────
w_device = widgets.Dropdown(
    options=['--- 선택하세요 ---', 'GPDO', 'MZM'],
    description='① 소자 종류',
    layout=dd_layout, style=style
)
w_wafer = widgets.Dropdown(
    options=['--- 먼저 소자를 선택하세요 ---'],
    description='② 웨이퍼',
    layout=dd_layout, style=style
)
w_ts = widgets.Dropdown(
    options=['--- 먼저 웨이퍼를 선택하세요 ---'],
    description='③ Timestamp',
    layout=dd_layout, style=style
)
w_die = widgets.Dropdown(
    options=['--- 먼저 Timestamp를 선택하세요 ---'],
    description='④ Die (col, row)',
    layout=dd_layout, style=style
)
out_status = widgets.Output()

# ── 파일 스캔 함수 ────────────────────────────────────
def scan_files(device, wafer, ts):
    """timestamp 폴더 내 해당 소자 XML 목록 반환 → {label: path}"""
    ts_dir = os.path.join(DATA_DIR, wafer, ts)
    result = {}
    for fname in sorted(os.listdir(ts_dir)):
        fpath = os.path.join(ts_dir, fname)
        if not fname.lower().endswith('.xml'):
            continue
        if device == 'GPDO' and GPDOParser.is_gpdo_xml(fpath):
            import re
            m = re.search(r'\((-?\d+),(-?\d+)\)', fname)
            label = f'({m.group(1)}, {m.group(2)})' if m else fname
            result[label] = fpath
        elif device == 'MZM' and MZMParser.MZM_FILENAME_TAG in fname:
            import re
            m = re.search(r'\((-?\d+),(-?\d+)\)', fname)
            label = f'({m.group(1)}, {m.group(2)})' if m else fname
            result[label] = fpath
    return result

# ── 콜백 ─────────────────────────────────────────────
def on_device_change(change):
    if change['new'] in ('--- 선택하세요 ---',):
        return
    dev = change['new']
    selection['device'] = dev
    wafers = DEVICE_WAFER_MAP[dev]
    w_wafer.options = ['--- 선택하세요 ---'] + wafers
    w_ts.options    = ['--- 먼저 웨이퍼를 선택하세요 ---']
    w_die.options   = ['--- 먼저 Timestamp를 선택하세요 ---']
    with out_status: clear_output(); print(f'✅ 소자: {dev}  →  웨이퍼를 선택하세요')

def on_wafer_change(change):
    if '선택' in change['new']: return
    wid = change['new']
    selection['wafer'] = wid
    wafer_dir = os.path.join(DATA_DIR, wid)
    tss = sorted([d for d in os.listdir(wafer_dir)
                  if os.path.isdir(os.path.join(wafer_dir, d))])
    w_ts.options  = ['--- 선택하세요 ---'] + tss
    w_die.options = ['--- 먼저 Timestamp를 선택하세요 ---']
    with out_status: clear_output(); print(f'✅ 웨이퍼: {wid}  →  Timestamp를 선택하세요')

def on_ts_change(change):
    if '선택' in change['new']: return
    ts = change['new']
    selection['timestamp'] = ts
    dev = selection['device']
    wid = selection['wafer']
    die_map = scan_files(dev, wid, ts)
    selection['_die_map'] = die_map
    if die_map:
        w_die.options = ['--- 선택하세요 ---', 'All (히트맵 + 박스플롯)'] + list(die_map.keys())
        with out_status: clear_output(); print(f'✅ Timestamp: {ts}  →  {len(die_map)}개 Die 발견. Die를 선택하세요')
    else:
        w_die.options = [f'❌ {dev} 파일 없음']
        with out_status: clear_output(); print(f'⚠ {ts} 에 {dev} 파일이 없습니다')

def on_die_change(change):
    if '선택' in change['new'] or '❌' in change['new']: return
    label = change['new']
    die_map = selection.get('_die_map', {})
    xml_path = die_map.get(label)
    selection['die']      = label
    selection['xml_path'] = xml_path
    with out_status:
        clear_output()
        print(f'━'*55)
        print(f'  ✅ 선택 완료!')
        print(f'  소자     : {selection["device"]}')
        print(f'  웨이퍼   : {selection["wafer"]}')
        print(f'  Timestamp: {selection["timestamp"]}')
        print(f'  Die      : {label}')
        print(f'  파일명   : {os.path.basename(xml_path)}')
        print(f'━'*55)
        print('  ▶ 아래 Cell 3을 실행하세요')

w_device.observe(on_device_change, names='value')
w_wafer.observe(on_wafer_change,   names='value')
w_ts.observe(on_ts_change,         names='value')
w_die.observe(on_die_change,       names='value')

display(widgets.VBox([w_device, w_wafer, w_ts, w_die, out_status]))

---
## Cell 3 — 데이터 불러오기 및 XML 파싱

In [12]:
# Cell 3 : 데이터 불러오기 및 XML 파싱
parsed_data_list = []  # ★ 매 실행마다 리셋

DEVICE_TYPE = selection.get('device')
WAFER_ID    = selection.get('wafer')
IS_ALL      = (selection.get('die') == 'All (히트맵 + 박스플롯)')
ts          = selection['timestamp']
ts_dir      = os.path.join(DATA_DIR, WAFER_ID, ts)

if not DEVICE_TYPE or not WAFER_ID or not ts:
    raise ValueError('Cell 2에서 소자 → 웨이퍼 → Timestamp → Die 를 순서대로 선택하세요.')

# 파싱할 XML 목록 결정
if IS_ALL:
    if DEVICE_TYPE == 'GPDO':
        xml_paths = sorted([
            os.path.join(ts_dir, f) for f in os.listdir(ts_dir)
            if f.lower().endswith('.xml') and GPDOParser.is_gpdo_xml(os.path.join(ts_dir, f))
        ])
    else:
        xml_paths = sorted([
            os.path.join(ts_dir, f) for f in os.listdir(ts_dir)
            if f.lower().endswith('.xml') and MZMParser.MZM_FILENAME_TAG in f
        ])
    print(f'[All 모드] {len(xml_paths)}개 파일 파싱 시작...')
else:
    xml_path = selection.get('xml_path')
    if not xml_path or not os.path.exists(xml_path):
        raise ValueError('Cell 2에서 Die 를 선택하세요.')
    xml_paths = [xml_path]

# 파싱 실행
for xml_path in xml_paths:
    fname = os.path.basename(xml_path)
    try:
        if DEVICE_TYPE == 'GPDO':
            raw = GPDOParser.parse(xml_path)
            entry = dict(timestamp=ts, fname=fname, xml_path=xml_path, raw=raw,
                         col=raw['col'], row=raw['row'],
                         lc_wl=raw['lc_wl'], fiber_dbm=raw['fiber_dbm'])
        else:
            data, root = MZMParser.parse_with_root(xml_path)
            entry = dict(timestamp=ts, fname=fname, xml_path=xml_path,
                         raw=data, root=root,
                         col=data.get('Column'), row=data.get('Row'))
        parsed_data_list.append(entry)
        print(f'  ✅ ({entry["col"]}, {entry["row"]})')
    except Exception as e:
        print(f'  ❌ {fname}: {e}')

print(f'\n파싱 완료: {len(parsed_data_list)}개 / {len(xml_paths)}개')


TypeError: join() argument must be str, bytes, or os.PathLike object, not 'NoneType'

---
## Cell 4 — 분석 결과 표 출력

In [ ]:
# Cell 4 : 피팅 + 결과 표
result_rows = []  # ★ 매 실행마다 리셋

for entry in parsed_data_list:
    raw = entry['raw']
    col = entry['col']; row = entry['row']; ts = entry['timestamp']

    if DEVICE_TYPE == 'GPDO':
        try:
            ref_r = FittingEngine.fit_reference(raw['L_ref'], raw['IL_ref'])
            df    = FittingEngine.fit_dark_fwd(raw['V_dark'], raw['I_dark'])
            dr    = FittingEngine.fit_dark_rev(raw['V_dark'], raw['I_dark'])
            lf    = FittingEngine.fit_light(raw['V_light'], raw['I_light'], df, dr)
            pc    = FittingEngine.calc_photo_current(
                        raw['V_light'], raw['I_light'], raw['V_dark'], raw['I_dark'])
            idx_wl = np.argmin(np.abs(raw['L_ref'] - raw['lc_wl']))
            resp   = FittingEngine.calc_responsivity(pc['Iph'], raw['fiber_dbm'], raw['IL_ref'][idx_wl])
            peak_wl = float(raw['L_spec'][np.argmax(np.abs(raw['I_spec']))]) if len(raw['L_spec']) > 0 else float('nan')
            entry['fit_results'] = dict(ref_r=ref_r, df=df, dr=dr, lf=lf, pc=pc, resp=resp, peak_wl=peak_wl)
            result_rows.append({
                'Die (Col,Row)': f'({col},{row})',
                'λc (nm)'      : f"{raw['lc_wl']:.1f}",
                'Fiber (dBm)'  : f"{raw['fiber_dbm']:.2f}",
                'Iph (A)'      : f"{pc['Iph']:.3e}",
                'n'            : f"{df['n']:.4f}",
                'R (A/W)'      : f"{resp['R_resp']:.4f}",
                'Peak λ (nm)'  : f"{peak_wl:.1f}" if not __import__('math').isnan(peak_wl) else 'N/A',
                'R² fwd'       : f"{df['r2']:.4f}",
            })
        except Exception as e:
            print(f'피팅 실패 [{entry["fname"]}]: {e}')
    else:
        result_rows.append({
            'Die (Col,Row)' : f'({col},{row})',
            'TestSite'      : raw.get('Testsite','N/A'),
            'Analysis λ'    : raw.get('Analysis Wavelength','N/A'),
            'I@-1V (A)'     : f"{raw['I at -1V [A]']:.3e}" if raw.get('I at -1V [A]') is not None else 'N/A',
            'Ideality n'    : f"{raw['Ideality Factor']:.4f}" if raw.get('Ideality Factor') is not None else 'N/A',
            'FSR (nm)'      : f"{raw['FSR (nm)']:.4f}" if raw.get('FSR (nm)') is not None else 'N/A',
            'ER (dB)'       : f"{raw['Extinction Ratio (dB)']:.2f}" if raw.get('Extinction Ratio (dB)') is not None else 'N/A',
            'R² Ref'        : f"{raw['Rsq of Ref. spectrum (Nth)']:.4f}" if raw.get('Rsq of Ref. spectrum (Nth)') is not None else 'N/A',
        })

display(pd.DataFrame(result_rows))


---
## Cell 5 — 그래프 / 히트맵 / 박스플롯

| 선택 | 출력 |
|------|------|
| 단일 Die | 그래프 4개 (기존 Plotter) |
| All | 히트맵 + Raincloud 박스플롯 |


In [ ]:
# Cell 5 : 그래프 시각화
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from unittest.mock import patch
from IPython.display import display as ipy_display
from src.boxplot import _raincloud_ax

IS_ALL = (selection.get('die') == 'All (히트맵 + 박스플롯)')

# ── _draw_and_save 의 ax 버전 헬퍼 ───────────────────
def _heatmap_on_ax(ax, cols, rows, vals, title, unit, cmap, wafer_id):
    valid   = [(int(c), int(r), float(v))
               for c, r, v in zip(cols, rows, vals)
               if v is not None and not np.isnan(float(v))]
    missing = [(int(c), int(r))
               for c, r, v in zip(cols, rows, vals)
               if v is None or np.isnan(float(v))]
    if not valid:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', color='gray')
        ax.set_title(title, fontweight='bold'); return
    cs, rs, vs = zip(*valid)
    norm = Normalize(vmin=min(vs), vmax=max(vs))
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    for c, r, v in zip(cs, rs, vs):
        rect = mpatches.FancyBboxPatch(
            (c-0.45, r-0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            facecolor=sm.to_rgba(v), edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        txt_color = 'white' if norm(v) < 0.5 else 'black'
        ax.text(c, r, f'{v:.3g}', ha='center', va='center',
                fontsize=8, fontweight='bold', color=txt_color)
    for c, r in missing:
        rect = mpatches.FancyBboxPatch(
            (c-0.45, r-0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            facecolor='#EEEEEE', edgecolor='gray',
            linewidth=1, linestyle='--')
        ax.add_patch(rect)
        ax.text(c, r, 'N/A', ha='center', va='center', fontsize=7, color='gray')
    all_c = list(cs) + [c for c,_ in missing]
    all_r = list(rs) + [r for _,r in missing]
    ax.set_xlim(min(all_c)-0.7, max(all_c)+0.7)
    ax.set_ylim(min(all_r)-0.7, max(all_r)+0.7)
    ax.set_aspect('equal')
    unit_str = f' [{unit}]' if unit else ''
    ax.set_xlabel('Column (Die X)', fontsize=9)
    ax.set_ylabel('Row (Die Y)', fontsize=9)
    ax.set_title(f'Wafer {wafer_id} – {title}{unit_str}', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.15, linestyle=':')
    plt.colorbar(sm, ax=ax, label=f'{title}{unit_str}', shrink=0.85)

# ════════════════════════════════════════════════
# ① 단일 Die — 기존 Plotter 그대로
# ════════════════════════════════════════════════
if not IS_ALL:
    captured = []
    def _capture(fig=None):
        if fig is not None: captured.append(fig)

    if DEVICE_TYPE == 'GPDO':
        from src.gpdo.plotter import Plotter as GPDOPlotter
        for entry in parsed_data_list:
            fr = entry.get('fit_results')
            if fr is None: print('Cell 4 를 먼저 실행하세요'); continue
            captured.clear()
            with patch('matplotlib.pyplot.close', side_effect=_capture):
                GPDOPlotter.plot(
                    entry['raw'], fr['ref_r'], fr['df'], fr['dr'], fr['lf'], fr['pc'], fr['resp'],
                    save_dir=None, wafer_id=WAFER_ID, fname=entry['fname'],
                )
            for fig in captured:
                ipy_display(fig); plt.close(fig)
    else:
        from src.mzm.plotter import Plotter as MZMPlotter
        for entry in parsed_data_list:
            if entry.get('root') is None: print('XML root 없음'); continue
            captured.clear()
            with patch('matplotlib.pyplot.close', side_effect=_capture):
                MZMPlotter.plot_from_root(
                    entry['root'], entry['fname'],
                    save_dir=None, verbose=True, extra_info=entry['raw'],
                )
            for fig in captured:
                ipy_display(fig); plt.close(fig)

# ════════════════════════════════════════════════
# ② All — 히트맵 가로 배치 + Raincloud 박스플롯
# ════════════════════════════════════════════════
else:
    cols_all = [e['col'] for e in parsed_data_list]
    rows_all = [e['row'] for e in parsed_data_list]

    if DEVICE_TYPE == 'GPDO':
        R_vals   = [e['fit_results']['resp']['R_resp'] if e.get('fit_results') else None for e in parsed_data_list]
        n_vals   = [e['fit_results']['df']['n']        if e.get('fit_results') else None for e in parsed_data_list]
        Iph_vals = [e['fit_results']['pc']['Iph']      if e.get('fit_results') else None for e in parsed_data_list]

        # 히트맵 2개 가로
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'GPDO Heatmap — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        _heatmap_on_ax(axes[0], cols_all, rows_all, R_vals,  'Responsivity',   'A/W', 'RdYlGn',  WAFER_ID)
        _heatmap_on_ax(axes[1], cols_all, rows_all, n_vals,  'Ideality Factor', '',   'coolwarm', WAFER_ID)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

        # Raincloud 3개 가로
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'GPDO Raincloud — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, vals, ylabel in zip(axes,
            [R_vals, n_vals, Iph_vals],
            ['Responsivity [A/W]', 'Ideality Factor', 'Photo Current [A]']):
            arr = np.array([float(v) for v in vals if v is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

    else:  # MZM
        rows_data = [e['raw'] for e in parsed_data_list]
        hm_params = [
            ('Max transmission of Ref. spec (dB)', 'Max Transmission', 'dB',  'RdYlGn'),
            ('I at -1V [A]',                       'I at -1V',         'A',   'RdYlBu'),
            ('Extinction Ratio (dB)',               'Extinction Ratio', 'dB',  'RdYlGn'),
            ('FSR (nm)',                            'FSR',              'nm',  'RdYlBu'),
        ]

        # 히트맵 4개 가로
        fig, axes = plt.subplots(1, 4, figsize=(26, 6))
        fig.suptitle(f'MZM Heatmap — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, (key, title, unit, cmap) in zip(axes, hm_params):
            vals = [r.get(key) for r in rows_data]
            _heatmap_on_ax(ax, cols_all, rows_all, vals, title, unit, cmap, WAFER_ID)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

        # Raincloud 4개 가로
        bp_params = [
            ('Extinction Ratio (dB)', 'Extinction Ratio [dB]'),
            ('FSR (nm)',              'FSR [nm]'),
            ('I at -1V [A]',         'I at -1V [A]'),
            ('Ideality Factor',      'Ideality Factor'),
        ]
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))
        fig.suptitle(f'MZM Boxplot — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, (key, ylabel) in zip(axes, bp_params):
            arr = np.array(
                [float(r[key]) for r in rows_data if r.get(key) is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

print('완료')


---
## Cell 7 (선택) — 웨이퍼 전체 일괄 분석

> ⚠ **주의:** 모든 웨이퍼 × 모든 소자 × 모든 파일을 처리하므로 **시간이 걸립니다.**  
> 특정 웨이퍼만 실행하려면 `TARGET_WAFERS` 리스트를 수정하세요.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Cell 7 : 웨이퍼 전체 일괄 분석 (run.py 파이프라인 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from run import run_device_wafer, main as run_all
from config import WAFER_IDS

# ── 실행 대상 설정 ─────────────────────────────────────
# 원하는 웨이퍼만 포함시키세요
TARGET_WAFERS  = ['D08', 'D23']   # 예: ['D07', 'D08', 'D23', 'D24'] 또는 WAFER_IDS
TARGET_DEVICES = ['GPDO']         # 'GPDO', 'MZM', 또는 둘 다

print(f'━'*55)
print(f'  일괄 분석 시작')
print(f'  대상 웨이퍼: {TARGET_WAFERS}')
print(f'  대상 소자  : {TARGET_DEVICES}')
print(f'━'*55)

batch_results = {}
for dtype in TARGET_DEVICES:
    for wid in TARGET_WAFERS:
        print(f'\n▶ {dtype} / {wid} 처리 중...')
        res = run_device_wafer(dtype, wid)
        batch_results.setdefault(dtype, {})[wid] = res
        if res:
            print(f'  ✅ 완료')
        else:
            print(f'  ⚠ 결과 없음 또는 실패')

print(f'\n━'*55)
print('  일괄 분석 완료. 결과는 res/ 폴더에 저장됩니다.')
print(f'━'*55)

# ── 요약 통계 출력 ─────────────────────────────────────
from src.gpdo.csv import save_total_csv as gpdo_save_total
from src.mzm.csv  import generate_total_csv as mzm_save_total

if 'GPDO' in TARGET_DEVICES:
    gpdo_csv_dir = os.path.join(RES_DIR, 'csv', 'GPDO', PROJECT_NAME)
    gpdo_save_total(gpdo_csv_dir)
    total_csv = os.path.join(gpdo_csv_dir, 'Total_Result.csv')
    if os.path.exists(total_csv):
        df_total = pd.read_csv(total_csv)
        print(f'\n▶ GPDO Total_Result.csv 통계 ({len(df_total)}행):')
        display(df_total.describe(include='all').T)

if 'MZM' in TARGET_DEVICES:
    mzm_save_total(verbose=True)
    total_csv = os.path.join(RES_DIR, 'csv', 'MZM', PROJECT_NAME, 'Total_Result.csv')
    if os.path.exists(total_csv):
        df_total = pd.read_csv(total_csv)
        print(f'\n▶ MZM Total_Result.csv 통계 ({len(df_total)}행):')
        display(df_total.describe(include='all').T)

---
## Cell 8 (선택) — Wafer-level 히트맵 시각화

> Cell 7 일괄 분석 후 생성된 Total CSV를 이용해 웨이퍼 맵을 출력합니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Cell 8 : Wafer-level 히트맵 (Total CSV 기반)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HEATMAP_DEVICE = 'GPDO'   # 'GPDO' 또는 'MZM'
HEATMAP_WAFER  = 'D08'    # 웨이퍼 ID

total_csv = os.path.join(RES_DIR, 'csv', HEATMAP_DEVICE, PROJECT_NAME,
                         f'{HEATMAP_WAFER}_Result.csv')

if not os.path.exists(total_csv):
    # 개별 웨이퍼 CSV 시도
    alt = os.path.join(RES_DIR, 'csv', HEATMAP_DEVICE, PROJECT_NAME, 'Total_Result.csv')
    if os.path.exists(alt):
        total_csv = alt

if not os.path.exists(total_csv):
    print(f'❌ CSV 파일 없음: {total_csv}')
    print('   먼저 Cell 7을 실행하거나, Cell 1~6으로 단일 웨이퍼를 분석해 주세요.')
else:
    df_hm = pd.read_csv(total_csv)
    wafer_data = df_hm[df_hm.get('Wafer', df_hm.get('wafer_id', df_hm.columns[0])) == HEATMAP_WAFER] \
                 if HEATMAP_WAFER in df_hm.values else df_hm

    print(f'  데이터: {len(wafer_data)}행, 컬럼: {list(wafer_data.columns)}')

    # 컬럼 자동 감지
    col_cand = [c for c in wafer_data.columns if 'col' in c.lower()]
    row_cand = [c for c in wafer_data.columns if 'row' in c.lower()]

    if HEATMAP_DEVICE == 'GPDO':
        param_list = [
            ('R_resp',  'Responsivity (A/W)',   'RdYlGn'),
            ('n_d',     'Ideality Factor',       'coolwarm'),
        ]
    else:
        param_list = [
            ('FSR (nm)',              'FSR (nm)',              'viridis'),
            ('Extinction Ratio (dB)', 'Extinction Ratio (dB)', 'RdYlGn'),
        ]

    for param_key, param_title, cmap_name in param_list:
        if param_key not in wafer_data.columns:
            print(f'  ⚠ 컬럼 없음: {param_key}  (건너뜀)')
            continue
        if not col_cand or not row_cand:
            print(f'  ⚠ col/row 컬럼 감지 실패')
            break

        col_c = col_cand[0]
        row_c = row_cand[0]
        sub = wafer_data[[col_c, row_c, param_key]].dropna()

        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(sub[col_c].astype(float), sub[row_c].astype(float),
                        c=sub[param_key].astype(float),
                        cmap=cmap_name, s=200, edgecolors='k', linewidths=0.5)
        plt.colorbar(sc, ax=ax, label=param_title)
        ax.set_xlabel('Column'); ax.set_ylabel('Row')
        ax.set_title(f'{HEATMAP_WAFER} — {param_title}  ({HEATMAP_DEVICE})')
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'  ✅ 히트맵 출력: {param_title}')